# ⚡ Concurrent Agent Workflows wit GitHub Models (Python)

## 📋 Advanced Parallel Processing Tutorial

Dis notebook dey show **concurrent workflow patterns** wey use Microsoft Agent Framework. You go learn how to build high-performance, parallel processing workflows wey get plenty AI agents wey dey run at di same time, dey improve throughput well well and dey enable sophisticated multi-threaded business processes.

## 🎯 Learning Objectives

### 🚀 **Concurrent Processing Fundamentals**
- **Parallel Agent Execution**: Make many agents run at di same time for maximum efficiency
- **Workflow Orchestration**: Arrange concurrent operations well well while still dey maintain data consistency
- **Performance Optimization**: Make big speedup through parallel processing
- **Resource Management**: Use AI model resources well well across concurrent operations

### 🏗️ **Advanced Concurrency Patterns**
- **Fork-Join Processing**: Break work among many agents and join results back
- **Pipeline Parallelism**: Places wey execution stages dey overlap for continuous throughput
- **Load Balancing**: Share work evenly across all available agent resources
- **Synchronization Points**: Arrange concurrent agents for important workflow stages

### 🏢 **Enterprise Concurrent Applications**
- **High-Volume Document Processing**: Process many documents at the same time
- **Real-Time Content Analysis**: Concurrent analysis of data wey dey come inside streams
- **Batch Processing Optimization**: Make throughput maximum for large-scale operations
- **Multi-Modal Analysis**: Parallel processing of different content types (text, images, data)

## ⚙️ Prerequisites & Setup

### 📦 **Required Dependencies**

Install Agent Framework wey get concurrent workflow capabilities:

```bash
pip install agent-framework-core -U
```

### 🔑 **GitHub Models Configuration**

**Environment Setup (.env file):**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

**Concurrent Processing Considerations:**
- **Rate Limits**: Watch GitHub Models API rate limits for concurrent requests
- **Resource Usage**: Think about memory and CPU use when many concurrent agents dey run
- **Error Handling**: Put strong error recovery for parallel operations

### 🏗️ **Concurrent Workflow Architecture**

```mermaid
graph TD
    A[Workflow Start] --> B[Concurrent Execution]
    B --> C[Agent Pool 1]
    B --> D[Agent Pool 2]
    B --> E[Agent Pool 3]
    C --> F[Result Aggregation]
    D --> F
    E --> F
    F --> G[Final Output]
    
    H[GitHub Models API] --> C
    H --> D
    H --> E
```

**Key Benefits:**
- **⚡ Performance**: Big speedup cause parallel execution
- **📈 Scalability**: Fit handle more work without time dey increase wahala
- **🔄 Efficiency**: Better use of available computational resources
- **🎯 Throughput**: Process more work for the same amount time

## 🎨 **Concurrent Workflow Design Patterns**

### 🔍 **Research & Analysis Pipeline**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Data Processing Workflow**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Content Creation Pipeline**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Multi-Stage Processing**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Enterprise Performance Benefits**

### ⚡ **Throughput Optimization**
- **Parallel Execution**: Many agents dey work together at di same time
- **Resource Utilization**: Maximum efficiency of available AI model capacity
- **Time Reduction**: Process time reduce well well
- **Scalable Architecture**: You fit add more concurrent agents anytime wey you need am

### 🛡️ **Reliability & Resilience**
- **Fault Tolerance**: If one agent fail, the whole workflow no go stop
- **Error Isolation**: Problem wey happen for one concurrent branch no go affect others
- **Graceful Degradation**: System go still dey work even if agent capacity reduce
- **Recovery Mechanisms**: Automatic retry and error handling for operations wey fail

### 📊 **Monitoring & Observability**
- **Concurrent Execution Tracking**: Track how all parallel work dey go
- **Performance Metrics**: Measure how speedup and efficiency dey improve
- **Resource Usage Analytics**: Make better plan for concurrent agent use
- **Bottleneck Identification**: Find wahala wey dey hold performance back and solve am

Make we build high-performance concurrent AI workflows! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = await provider.create_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = await provider.create_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)

In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
